In [1]:
import json
from pathlib import Path
import pandas as pd
import re
from scripts import extract as ext

In [2]:
ROOT = Path("~/code/selector_examenes").expanduser()

In [3]:
with open(ROOT / "diccionarios_clasificacion_ejercicios/quimica/temas_ejercicio.json", "r") as file:
    temas_ejercicio = json.load(file)

with open(ROOT / "diccionarios_clasificacion_ejercicios/quimica/quimica_keywords.json") as file:
    quimica_keywords = json.load(file)

In [4]:
def classify_exercise(exam_year: int, exam_name: str, statement: str, keywords = quimica_keywords) -> tuple:
    # obtener el año del examen
    exam_name = (" - ".join(exam_name.split("_")[1:])).capitalize()

    # convertir saltos de línea en espacios
    statement = re.sub(r"\s+", " " , statement)
    
    # seleccionar modelo de examen en función del año
    if exam_year <= 2019:
        exam_model = "modelo_2010_2019"
        exercise_number_pattern = r"^\d{1}\.-"

    elif exam_year <= 2024:
        exam_model = "modelo_2020_2024"
        exercise_number_pattern = r"^[ABC]\d{1}"
    else:
        exam_model = "modelo_2025+"
        exercise_number_pattern = r"^\d{1}?[AB]"

    # temas del ejercicio según número ejercicio y modelo

    # "modelo_2024+": {
    #     "tipo_numeración" : "1A, 1B, 2A, 2B, ..., 4B, 5",
    #     "1": ["configuracion_electronica", ...]
    #     "2": ["equilibrio_quimico", ...],

    exercise_topics_dict = temas_ejercicio[exam_model]

    # obtener el número del ejercicio
    exercise_number = re.match(exercise_number_pattern, statement)

    # si se encuentra el número de ejercicio
    if exercise_number:

        # extraer la clave del diccionario del modelo de examen
        if exam_year <= 2019:  # 2010 - 2019 - key 1,2,3,...
            exercise_number_key = exercise_number[0][0]

            # las keys son: "1", "2,3,4", "5,6"
            # exercise_number_key necesita un ajuste
            # para que se encuentre la clave
            for key in exercise_topics_dict.keys():
                if exercise_number_key in key:
                    exercise_number_key = key
                    break
        
        elif 2020 <= exam_year <= 2024:  # 2020 - 2024 - key A, B, C
            exercise_number_key = exercise_number[0][0]

        else:  # 2025 + - key 1, 2, 3, 4A, 4B, 5
            exercise_number_key = exercise_number[0]
            if exercise_number[0] not in  ["4A", "4B"]:
                exercise_number_key = exercise_number[0][0]
            else:
                exercise_number_key = exercise_number[0]

        # limpiar el número de ejercicio
        exercise_number = re.sub(r"[\.-]", "", exercise_number[0])

        # ejercicios de formulación - NO CLASIFICAR

        # condiciones a cumplir para ser ejercicio de formulación
        formulation_pre_2020 = exam_year <= 2019 and exercise_number_key == "1"
        formulation_2020_2024 =  (2020 <= exam_year <= 2024) and exercise_number_key == "A"

        if formulation_pre_2020 or formulation_2020_2024:
            exercise_topics = ["Formulación"]
            exercise_types = ["Formulación"]

        elif exam_year >= 2025 and exercise_number_key == "4A":
            exercise_topics = ["Formulación"]
            exercise_types = ["Formulación orgánica"]

        else:

            ###### CLASIFICACIÓN #######
            
            # opciones de tema para el ejercicio suministrado
            exercise_topic_options = exercise_topics_dict[exercise_number_key]

            # recorro los posibles temas del ejercicio
            # e intento clasificar el tema del ejercicio
            exercise_topics = []
            for topic in exercise_topic_options:

                translation_table = str.maketrans({"á" : "a", "é" : "e", "í" : "i", "ó" : "o", "ú" : "u"})
                topic = topic.translate(translation_table)
                
                # obtengo las palabras clave que identifican el tema
                # y los tipos de ejercicios del tema y sus respectivas keywords
                
                unique_topic_keywords = list(keywords[topic]["keywords_tema"])
                for key, value in keywords[topic]["tipos_ejercicio"].items():
                    unique_topic_keywords.extend(value)

                pattern = re.compile("|".join(unique_topic_keywords), flags=re.IGNORECASE | re.DOTALL)
                match_topic = re.search(pattern, statement)
                if match_topic:
                    exercise_topics.append(topic)

            # lista de tipos de ejercicio
            exercise_types = []

            # si se ha identificado el tema
            # lo siguiente es asignar tipo(s) de ejercicio
            if len(exercise_topics) > 0:
                # compruebo existencia de keywords de cada tipo de ejercicio en el enunciado
                for topic in exercise_topics:
                    exercise_type_keywords = keywords[topic]["tipos_ejercicio"]
                    for tipo, tipo_keywords in exercise_type_keywords.items():
                        pattern = re.compile("|".join(tipo_keywords), flags=re.IGNORECASE | re.DOTALL)
                        match_type = re.search(pattern, statement)
                        if match_type:
                            print(f"{exercise_number}: tema {exercise_topics}, tipo: {match_type[0]}")
                            exercise_types.append(tipo)
            # no se ha clasificado ni el tema ni el tipo de ejercicio            
            else:
                exercise_topics, exercise_types = [], [] 
    else:
        exercise_topics, exercise_types = [], []

    return (exam_year, exam_name, exercise_number, exercise_topics, exercise_types, statement)

In [5]:
def classify_exam_period(exam_year):
    if exam_year <= 2019:
        exam_period = "2010-2019"
    elif exam_year <= 2024:
        exam_period = "2020-2024"
    else:
        exam_period = "2025+"

    return exam_period

## Test con un archivo

In [6]:
file = ROOT / "examenes_quimica/2022_quimica/química - 2022 - reserva 1.pdf"

In [7]:
exam_text = ext.extract_exercises_from_pdf(file)

exam_year = int(re.search(r"\d{4}", file.name)[0])
exam_period = classify_exam_period(exam_year)

exercises_list = ext.split_exercises(exam_text, exam_period=exam_period)

unclassified_exercises_df = pd.DataFrame([{"year" : exam_year, "exam_option" : exam_option, "statement" : st} for exam_option, st in exercises_list])

In [8]:
# unclassified_exercises_df

Enunciado de un ejercicio

In [9]:
# print(exercises_list[11][1])

Clasificar ejercicio

In [10]:
df_list = []
for ex in exercises_list:
    statement = ex[1]
    year = 2022
    exam_name = file.stem
    year, exam_name, ex_num, topic, exercise_types, st = classify_exercise(year, exam_name, statement, quimica_keywords)

    df_list.append({"year": year, "exam_name": exam_name,
                    "exercise_number": ex_num, "topic": topic,
                    "exercise_types": exercise_types, "statement": st})
df = pd.DataFrame(df_list)

B1: tema ['configuracion_electronica', 'redox'], tipo: iones más estables
B1: tema ['configuracion_electronica', 'redox'], tipo: desaparea
B2: tema ['configuracion_electronica', 'propiedades_periodicas', 'redox'], tipo: desaparea
B2: tema ['configuracion_electronica', 'propiedades_periodicas', 'redox'], tipo: radio
B3: tema ['enlace_quimico', 'equilibrio_quimico', 'acido_base', 'redox', 'reactividad_organica'], tipo: dureza
B3: tema ['enlace_quimico', 'equilibrio_quimico', 'acido_base', 'redox', 'reactividad_organica'], tipo: longitud del enlace
B3: tema ['enlace_quimico', 'equilibrio_quimico', 'acido_base', 'redox', 'reactividad_organica'], tipo: fuerzas
B3: tema ['enlace_quimico', 'equilibrio_quimico', 'acido_base', 'redox', 'reactividad_organica'], tipo: disminuye
B3: tema ['enlace_quimico', 'equilibrio_quimico', 'acido_base', 'redox', 'reactividad_organica'], tipo: ha
B3: tema ['enlace_quimico', 'equilibrio_quimico', 'acido_base', 'redox', 'reactividad_organica'], tipo: longitud de

In [20]:
df

,year,exam_name,exercise_number,topic,exercise_types,statement
0,2022,,A1,[Formulación],[Formulación],A1. Formule o nombre los siguientes compuestos...
1,2022,,A2,[Formulación],[Formulación],A2. Formule o nombre los siguientes compuestos...
2,2022,,B1,"[configuracion_electronica, redox]","[Ion más estable, electrones desapareados, aju...","B1. Los elementos A, B, C y D tienen números a..."
3,2022,,B2,"[configuracion_electronica, propiedades_period...","[electrones desapareados, Propiedades periódicas]","B2. Considerando los siguientes elementos Mg, ..."
4,2022,,B3,"[enlace_quimico, equilibrio_quimico, acido_bas...","[Comparación Born-Haber, Hibridación carbono, ...",B3. Justifique: a) ¿Qué compuesto tendrá mayor...
5,2022,,B4,"[redox, reactividad_organica]","[pilas galvánicas, notación de la pila (se pre...",B4. Se construye una pila galvánica formada po...
6,2022,,B5,"[enlace_quimico, equilibrio_quimico, solubilid...",[Solubilidad compuestos covalentes moleculares...,B5. Dadas las siguientes especies con sus prod...
7,2022,,B6,"[redox, reactividad_organica]","[Reacciones orgánicas, Isomería]",B6. Dados los siguientes compuestos: CH₃COCH₂O...
8,2022,,C1,"[equilibrio_quimico, acido_base, redox]","[Principio de Le Chatelier, Cálculo de Kc y Kp...","C1. Se introducen 0,035 moles de I₂ en un reci..."
9,2022,,C2,"[solubilidad, redox]","[efecto del ion común - cálculo, cálculo de so...","C2. A 25 ºC, la constante de solubilidad del A..."


In [93]:
for idx, row in df.iterrows():
    print(f"Tema: {row.topic}")
    print("-" * 20)
    print(row.statement)
    print("\n\n")

Tema: Formulación
--------------------
A1. Formule o nombre los siguientes compuestos: a) Sulfuro de manganeso(III); b) Fosfato de aluminio; c) 1,2-Diclorobenceno; d) CrO₃; e) MgH₂; f) CH₃CH₂COOCH₃ 



Tema: Formulación
--------------------
A2. Formule o nombre los siguientes compuestos: a) Hidróxido de cobalto(II); b) Hidrogenocarbonato de magnesio; c) Metilbenceno; d) MoO₃; e) Ni₂Se₃; f) CH₂OHCH₂CH₂COOH 



Tema: configuracion_electronica
--------------------
B1. Los elementos A, B, C y D tienen números atómicos 12, 14, 17 y 37, respectivamente. a) Escriba la configuración electrónica de B y D. b) Indique los iones más estables de A y C y escriba la configuración electrónica de cada uno de ellos. c) Indique cuál o cuáles de los elementos tienen electrones desapareados en su estado fundamental. 



Tema: configuracion_electronica
--------------------
B2. Considerando los siguientes elementos Mg, Si y Cl, justifique: a) Cuál de ellos tiene mayor radio. b) Cuál de ellos tiene mayor tend

In [45]:
classify_exercise(2024, file.stem, exercises_list[11][1], quimica_keywords)

(2024,
 'Quimica - química - examen - suplente-a - química',
 'C4',
 'acido_base',
 ['disoluciones'],
 'C4. El níquel metálico reacciona con ácido nítrico concentrado según la reacción: Ni + HNO₃ → Ni(NO₃)₂ + NO + H₂O a) Ajuste las ecuaciones iónica y molecular por el método del ion-electrón. b) Calcule la masa de níquel que podrá oxidarse con 1 mL de ácido nítrico comercial del 70% de riqueza en masa y densidad 1,42 g·mL⁻¹. Datos: Masas atómicas relativas: Ni= 58,7; N= 14; O= 16; H=1')

## Clasificar los ejercicios de un año

In [36]:
# classifications = []
# for file in (ROOT / "examenes_quimica/2024_quimica").iterdir():
#     if file.is_file():
#         exam_text = ext.extract_exercises_from_pdf(file)
#         exam_year = int(file.name.split("_")[0])

#         if exam_year < 2020:
#             exam_period = "2010-2019"
#         elif exam_year <= 2024:
#             exam_period = "2020-2024"
#         else:
#             exam_period = "2025+"

#         exercises_list = ext.split_exercises(exam_text, exam_period=exam_period)

#         # exercises_list devuelve una tupla: (exam_option, statement)
#         for exercise_tuple in exercises_list:
#             try:
#                 statement = exercise_tuple[1]
#                 exam_option = exercise_tuple[0]
#                 classification = classify_exercise(file.stem, statement, keywords=quimica_keywords)
#                 classification = list(classification)
#                 classification.append(exam_option)
#                 classifications.append(classification)
#             except Exception as e:
#                 print(f"Error: file {file.name}. Exception: {e}")

# df_year = pd.DataFrame(classifications, columns = ["year", "exam_name", "exercise_number", "topic", "exercise_type", "statement", "exam_option"])
# df_year = df_year.explode("exercise_type", ignore_index=True)

In [37]:

# df_year = pd.DataFrame(classifications, columns = ["year", "exam_name", "exercise_number", "topic", "exercise_type", "statement", "exam_option"])
# df_year = df_year.explode("exercise_type", ignore_index=True)

In [38]:
# df_year.head()

## Clasificar todos los exámenes

In [41]:
classifications = []
exam_files = sorted((ROOT / "examenes_quimica").rglob("*.pdf"))

for file in exam_files:
    #try:
    if file.is_file():
        exam_text = ext.extract_exercises_from_pdf(file)
        exam_year = int(re.search(r"\d{4}", file.name)[0])

        exam_period = classify_exam_period(exam_year)

        exercises_list = ext.split_exercises(exam_text, exam_period=exam_period)

        # exercises_list devuelve una tupla: (exam_option, statement)
        for exercise_tuple in exercises_list:
            #try:
            statement = exercise_tuple[1].strip()
            exam_option = exercise_tuple[0]
            classification = classify_exercise(exam_year, file.stem, statement)
            classification = list(classification)
            classification.append(exam_option)
            classification.append(file.relative_to(ROOT))
            
            classifications.append(classification)
    #         except Exception as e:
    #             print(f"Error: file {file.name}. Exception: {e}")
    # except Exception as e:
    #     print(f"{file.name}. Error: {e}")

In [42]:

df_classified = pd.DataFrame(classifications, columns = ["year", "exam_name", "exercise_number", "topic", "exercise_type", "statement", "exam_option", "exam_file_path"])
df_classified = df_classified.explode("exercise_type", ignore_index=True)

In [43]:
df_classified.head()

,year,exam_name,exercise_number,topic,exercise_type,statement,exam_option,exam_file_path
0,2010,,1,Formulación,Formulación,1.- Formule o nombre los compuestos siguientes...,Opción A,examenes_quimica/2010_quimica/química - 2010 -...
1,2010,,2,configuracion_electronica,Isoelectrónicos,2.- Supongamos que los sólidos cristalinos NaF...,Opción A,examenes_quimica/2010_quimica/química - 2010 -...
2,2010,,3,configuracion_electronica,Isoelectrónicos,3.- Se dispone de una disolución acuosa satura...,Opción A,examenes_quimica/2010_quimica/química - 2010 -...
3,2010,,4,configuracion_electronica,verdadero o falso,4.- Razone si las siguientes afirmaciones son ...,Opción A,examenes_quimica/2010_quimica/química - 2010 -...
4,2010,,4,configuracion_electronica,Isoelectrónicos,4.- Razone si las siguientes afirmaciones son ...,Opción A,examenes_quimica/2010_quimica/química - 2010 -...


In [48]:
df_classified.topic.unique()

<ArrowStringArray>
[              'Formulación', 'configuracion_electronica',
        'equilibrio_quimico',                         nan,
              'termoquimica',      'reactividad_organica']
Length: 6, dtype: str

In [46]:
df_classified.isna().sum()

year                 0
exam_name            0
exercise_number     12
topic              198
exercise_type      213
statement            0
exam_option        747
exam_file_path       0
dtype: int64

In [44]:
df_classified.to_csv(ROOT / "CSV/df_classified.csv", index=False, encoding="utf-8")

In [ ]:
# df_classified[df_classified.topic == "Formulación"].to_csv("assign_exam_names/csv/form_classified.csv", index=False, encoding="utf-8")

In [45]:
df_classified.year.aggregate(["min", "max"])

min    2010
max    2026
Name: year, dtype: int64